<a href="https://colab.research.google.com/github/sinbbang/L-IntelligentSystem/blob/main/%EC%A7%80%EB%8A%A5%EC%8B%9C%EC%8A%A4%ED%85%9C_02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 단어 쌍의 등장 횟수를 쉽게 계산하기 위한 Counter
from collections import Counter


# 현재 vocabulary에서 서로 이웃한 문자(토큰) 쌍의 빈도를 계산
def get_stats(vocab):
    pairs = Counter()

    # vocab의 각 단어와 등장 횟수를 하나씩 확인
    for word, freq in vocab.items():

        # 공백을 기준으로 문자를 분리
        # 예: "l o w </w>" → ["l", "o", "w", "</w>"]
        symbols = word.split()

        # 서로 이웃한 문자 쌍을 확인
        # 예: (l, o), (o, w), (w, </w>)
        for i in range(len(symbols) - 1):

            # 해당 문자 쌍의 빈도에 단어의 등장 횟수를 더함
            pairs[(symbols[i], symbols[i + 1])] += freq

    return pairs


# 선택된 문자(토큰) 쌍을 하나의 토큰으로 합치는 함수
def merge_pair(pair, vocab):

    # 예: ('e', 's')
    # old = "e s", new = "es"
    old = " ".join(pair)
    new = "".join(pair)

    # vocabulary의 모든 단어에서 해당 쌍을 하나로 합침
    return {
        w.replace(old, new): f
        for w, f in vocab.items()
    }


# 초기 vocabulary
# 각 문자를 공백으로 분리해서 표현
# </w>는 단어의 끝을 나타내는 기호
vocab = {
    "l o w </w>": 5,        # low가 5번 등장
    "l o w e r </w>": 2,    # lower가 2번 등장
    "n e w e s t </w>": 6,  # newest가 6번 등장
    "w i d e s t </w>": 3   # widest가 3번 등장
}


# BPE 병합 과정을 5번 반복
for step in range(5):
    pairs = get_stats(vocab)

    if not pairs:
        break

    # 가장 자주 등장하는 토큰 쌍 선택
    best, freq = pairs.most_common(1)[0]

    # 선택된 토큰 쌍을 하나로 합침
    vocab = merge_pair(best, vocab)

    print(step + 1, best, freq)


# 최종 토큰 집합 확인
tokens = set()

for word in vocab:
    tokens.update(word.split())

print("\n최종 토큰 집합:")
print(sorted(tokens))

In [ ]:
# 정규표현식과 유니코드 처리를 위한 라이브러리
import re
import unicodedata


# 텍스트를 정제하는 함수
def clean_text(text: str) -> str:

    # 유니코드 문자를 표준 형태(NFC)로 통일
    # 예: 서로 다른 방식으로 표현된 한글을 같은 형태로 정규화
    text = unicodedata.normalize("NFC", text)

    # HTML 태그 제거
    # 예: <p>안녕하세요</p> → 안녕하세요
    text = re.sub(r"<[^>]+>", " ", text)

    # 인터넷 주소(URL)를 <URL>로 변경
    # 예: https://example.com → <URL>
    text = re.sub(r"https?://\S+|www\.\S+", " <URL> ", text)

    # 이메일 주소를 <EMAIL>로 변경
    # 예: student@example.com → <EMAIL>
    text = re.sub(
        r"[\w.+-]+@[\w-]+(?:\.[\w-]+)+",
        " <EMAIL> ",
        text
    )

    # 전화번호를 <PHONE>으로 변경
    # 예: 053-123-4567 → <PHONE>
    text = re.sub(
        r"0\d{1,2}[-.\s]?\d{3,4}[-.\s]?\d{4}",
        " <PHONE> ",
        text
    )

    # 연속된 공백이나 탭을 하나의 공백으로 변경
    text = re.sub(r"[ \t]+", " ", text)

    # 줄바꿈이 3번 이상 연속되면 2번으로 줄임
    text = re.sub(r"\n{3,}", "\n\n", text)

    # 문자열 앞뒤의 불필요한 공백 제거
    return text.strip()

In [ ]:
# 한국어 예제
# 정제할 원본 텍스트
raw = "<p>문의는 student@example.com 또는 053-123-4567 로 주세요.</p>"

# 텍스트 정제 결과 출력
print(clean_text(raw))

In [ ]:
# 중국어 예제
# 뜻: 문의 사항이 있으면 student@example.com 또는 010-1234-5678로 연락해 주세요.
raw_zh = "<p>如有问题，请联系 student@example.com 或 010-1234-5678。</p>"

# 정제 결과 출력
print("\n[중국어]")
print(clean_text(raw_zh))

In [ ]:
# Hugging Face의 토크나이저를 불러오기 위한 라이브러리
from transformers import AutoTokenizer


# 사용할 사전학습 모델 이름
# 여러 언어를 지원하는 Multilingual BERT 모델
model_id = "bert-base-multilingual-cased"


# 해당 모델에서 사용하는 토크나이저 불러오기
tok = AutoTokenizer.from_pretrained(model_id)


# 토큰화할 한국어 문장
text = "자연어처리는 정말 흥미롭습니다."


# 문장을 토큰 단위로 분리
# 예: ['자연', '##어', ...]와 같은 형태
tokens = tok.tokenize(text)


# 문장을 토큰 ID(숫자)로 변환
# add_special_tokens=False:
# [CLS], [SEP] 같은 특수 토큰은 추가하지 않음
ids = tok.encode(
    text,
    add_special_tokens=False
)


# 토큰 ID를 다시 문자열로 변환
restored = tok.decode(ids)


# 분리된 토큰 출력
print("tokens :", tokens)

# 각 토큰에 해당하는 숫자 ID 출력
print("ids    :", ids)

# 토큰 ID를 다시 문자열로 복원한 결과 출력
print("decode :", restored)

# 토크나이저가 가지고 있는 전체 어휘의 크기 출력
print("vocab  :", tok.vocab_size)

In [ ]:
# 한국어 문장
text_ko = "자연어처리는 정말 흥미롭습니다."

# 중국어 문장
# 뜻: 자연어 처리는 정말 흥미롭습니다.
text_zh = "自然语言处理真的很有趣。"


# 한국어 토큰화
tokens_ko = tok.tokenize(text_ko)
ids_ko = tok.encode(text_ko, add_special_tokens=False)

# 중국어 토큰화
tokens_zh = tok.tokenize(text_zh)
ids_zh = tok.encode(text_zh, add_special_tokens=False)


# 결과 출력
print("[한국어]")
print("tokens :", tokens_ko)
print("ids    :", ids_ko)
print("decode :", tok.decode(ids_ko))

print("\n[중국어]")
print("tokens :", tokens_zh)
print("ids    :", ids_zh)
print("decode :", tok.decode(ids_zh))